# v5同状态等beta三臂费后数值标签

冻结动作+0.10名义beta，Qlib一分钟`t+1`假定可用、`t+2`整股成交，至下一开盘。SOXL、SOXX、HOLD从同一父本状态独立计Tiered费用；未成交/恢复/终端未知。仅标签与支持，不拟合新特征或开候选账户。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import hashlib,json,warnings
from copy import deepcopy
from pathlib import Path
import exchange_calendars as xc
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.workflow.minute_action_value import _source
from src.backtest.event_guard_clock import guard_slot
from src.analysis.stage1_v5_equal_beta_value_labels import build_labels
from qlib.workflow import R
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
root=Path.cwd();card=root/"research/stage1_v5_equal_beta_value_labels_20261030"
raw_plan=(card/"plan.json").read_bytes();plan=json.loads(raw_plan)
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan["provider_uri"]/"qlib_store_manifest.json")==plan["provider_manifest_sha256"]
assert sha(root/plan["label_code"])==plan["label_code_sha256"]
qlib.init(provider_uri=str(root/plan["provider_uri"]),region="us",kernels=1)
source,source_receipt=_source(root,plan["training_source"],["analysis/native/dated_f696_0"])
frame_source,frame_receipt=_source(root,plan["frame_source"],["inputs/covered_frame"])
book=source["analysis/native/dated_f696_0"];frame=frame_source["inputs/covered_frame"].reset_index(drop=True)
assert len(frame)==plan["expected_parent_sessions"]==len(book["authority"])
assert frame.trade_date.equals(book["authority"].trade_date)
assert book["audit"]["status"]=="passed" and book["audit"]["slippage_bps"]==0
bars=D.features(["SOXL","SOXX"],["$open","$close","$raw_open","$price_valid"],start_time=plan["training_start"]+" 09:31:00",end_time=plan["training_end"]+" 16:00:00",freq="1min")
parts=[]
for symbol in ("SOXL","SOXX"):
    part=bars.xs(symbol,level="instrument").copy()
    part.columns=[symbol+"_"+field.removeprefix("$") for field in part]
    parts.append(part)
quotes=pd.concat(parts,axis=1).sort_index()
schedule=xc.get_calendar("XNYS",start=frame.trade_date.min(),end=frame.trade_date.max()).schedule
closes={pd.Timestamp(day).tz_localize(None):pd.Timestamp(row.close).tz_convert("America/New_York").tz_localize(None) for day,row in schedule.iterrows()}


In [2]:
if (card/"summary.json").exists():
    from src.core.recorder_sources import load_verified_recorder_objects
    summary=json.loads((card/"summary.json").read_text())
    assert summary["plan_sha256"]==hashlib.sha256(raw_plan).hexdigest()
    manifest_sha=sha(card/"verification_manifest.json")
    saved,receipt=load_verified_recorder_objects(card,expected_recorder_id=summary["recorder_id"],
        artifacts=["analysis/labels","analysis/annual","analysis/audit","analysis/summary"],
        expected_manifest_sha256=manifest_sha)
    assert len(saved["analysis/labels"])==saved["analysis/audit"]["event_slots"]
    print("recovered_verified_recorder",summary["recorder_id"],saved["analysis/audit"]["paired_rows"])
else:
    
    labels,masks,audit=build_labels(frame,book,quotes,closes=closes,tariff=plan["tariff"],delta_beta=.10)
    assert audit["sessions"]==plan["expected_parent_sessions"] and audit["event_slots"]<=plan["budget"]["maximum_event_slots"]
    assert audit["parent_nav_max_error_usd"]<=1e-7
    paired=labels.loc[labels.soxl_funded & labels.soxx_funded].copy()
    paired["year"]=paired.trade_date.dt.year
    perday=paired.groupby(["trade_date","year"],observed=True).relative_value_soxl_minus_soxx.agg(
        rows="size",any_positive=lambda x:bool(x.gt(0).any()),any_negative=lambda x:bool(x.lt(0).any())).reset_index()
    annual=perday.groupby("year",sort=True).agg(days=("trade_date","size"),paired_rows=("rows","sum"),
        positive_days=("any_positive","sum"),negative_days=("any_negative","sum")).reset_index()
    counts={int(row.year):int(row.days) for row in annual.itertuples(index=False)}
    sign={int(row.year):(int(row.positive_days),int(row.negative_days)) for row in annual.itertuples(index=False)}
    gate=plan["support_gate"]
    passed=bool(sum(counts.get(y,0) for y in (2018,2019,2020))>=gate["minimum_discovery_funded_days"]
        and counts.get(2019,0)>=gate["minimum_2019_days"] and counts.get(2020,0)>=gate["minimum_2020_days"]
        and counts.get(2021,0)>=gate["minimum_2021_days"] and counts.get(2022,0)>=gate["minimum_2022_days"]
        and all(min(sign.get(y,(0,0)))>=gate["minimum_positive_and_negative_days_2019_2022"]
                for y in (2019,2020,2021,2022)))
    tracking=card/"qlib_recorder"
    with R.start(experiment_name=card.name,uri=str(tracking)):
        recorder=R.get_recorder()
        recorder.save_objects(artifact_path="inputs",plan=plan,source_audits={"parent":source_receipt,"frame":frame_receipt})
        recorder.save_objects(artifact_path="sources",label_builder=(root/plan["label_code"]).read_text())
        recorder.save_objects(artifact_path="analysis",labels=labels,masks=masks,perday=perday,annual=annual,audit=audit)
        recorder.log_metrics(paired_rows=float(audit["paired_rows"]),paired_days=float(audit["paired_days"]),sign_support_gate_pass=float(passed))
    summary={"experiment":plan["experiment"],"plan_sha256":hashlib.sha256(raw_plan).hexdigest(),
        "decision":"fixed_010_value_label_sign_support_pass_needs_control_residual" if passed else "not_promoted_fixed_010_value_label_sign_support_failed",
        "historical_research_passed":False,"objective_achieved":False,"source_audit":audit,"support_gate_pass":passed,
        "annual_sign_support":annual.to_dict("records"),"new_model_fits":0,"new_native_accounts":0,
        "recorder_id":recorder.id,"experiment_id":str(recorder.experiment_id),"recorder_uri":str(tracking),
        "recorder_local_dir":str(tracking/str(recorder.experiment_id)/recorder.id),
        "artifact_policy":{"policy":"slim_card_recorder_first","workflow_config":"workflow.yaml","recorder_only_artifacts":["inputs/*","sources/*","analysis/*"]},
        "limitations":["Labels are one-event, one-next-open local value, not a complete own-state candidate strategy.",
            "Original recovery days are retrospectively masked, not a decision-time feature.",
            "Historical minute first receipt and broker fill remain unqualified.",
            "2021/2022 labels are retained for later fixed information tests; no candidate feature was selected."]}
    (card/"annual_support.csv").write_text(annual.to_csv(index=False))
    (card/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n")
    (card/"workflow.yaml").write_text("experiment: "+plan["experiment"]+"\nnotebook: "+plan["experiment"]+".ipynb\nrecorder_uri: "+str(tracking)+"\nartifact_policy: slim_card_recorder_first\n")
    print(json.dumps({"decision":summary["decision"],"source_audit":audit,"annual":summary["annual_sign_support"],"recorder_id":recorder.id},ensure_ascii=False))

## 中文结论

冻结+0.10名义beta三臂，在Qlib分钟源的384660事件槽上，同一分钟SOXL和SOXX各自整股费后成交且到下一开盘有效的成对价值标签333631条、覆盖1044交易日；2018–2022分别53/252/242/252/245日，逐年正负日支持门通过。父本HOLD下一开盘NAV最大误差1.16e-10美元。首轮计算后仅指标记录处float(dict)出错；保存标签SHA不变，同一Recorder恢复并完成指标、summary，见repair_receipt.json。标签不是模型/完整账户超额；2021/2022留作后续固定信息检验。原恢复日的事后掩码不能作决策特征，历史真实分钟首收和成交回执未知。NAS上传及labels/audit限定恢复见nas_binding.json、nas_scoped_readback.json。